In [1]:
import pandas as pd
import numpy as np
import json
import pickle

df = pd.read_csv('faceit_clean.csv')

separating features  
separating output for classification model and regression model

In [2]:
label_cols = ['team1_win', 'score_diff']
cat_cols = ['map']
num_cols = [c for c in df.columns if c not in label_cols + cat_cols]

X = df[num_cols + cat_cols]
y = df['team1_win']

# Train / Test split

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Preprocessing Pipeline

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ]
)

LogisticRegression model

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

log_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_pipe.fit(X_train, y_train)

y_pred_lr = log_reg_pipe.predict(X_test)
y_proba_lr = log_reg_pipe.predict_proba(X_test)[:, 1]

cv_lr = cross_val_score(log_reg_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(accuracy_score(y_test, y_pred_lr))

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


0.7695


RandomForest

In [7]:
from sklearn.ensemble import RandomForestClassifier


In [8]:
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
])

rf_pipe.fit(X_train, y_train)

y_pred_rf = rf_pipe.predict(X_test)
y_proba_rf = rf_pipe.predict_proba(X_test)[:, 1]

cv_rf = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f'Random Forest:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_rf):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_rf):.3f}')
print(f'CV:        {cv_rf.mean():.3f} (+/- {cv_rf.std():.3f})')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Random Forest:
  Accuracy:  0.772
  F1 Score:  0.777
  AUC:       0.873
  CV:        0.754 (+/- 0.013)


GradientBoostingClassifier

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

In [10]:
gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
])

gb_pipe.fit(X_train, y_train)

y_pred_gb = gb_pipe.predict(X_test)
y_proba_gb = gb_pipe.predict_proba(X_test)[:, 1]
cv_gb = cross_val_score(gb_pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f'Gradient Boosting:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_gb):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_gb):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_gb):.3f}')
print(f'CV:        {cv_gb.mean():.3f} (+/- {cv_gb.std():.3f})')

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Gradient Boosting:
  Accuracy:  0.786
  F1 Score:  0.792
  AUC:       0.892
  CV:        0.772 (+/- 0.014)


# Neural Network

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.callbacks import EarlyStopping

preparing data

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [15]:
input_dim = X_train_processed.shape[1]

model_clf = Sequential()
model_clf.add(Dense(128, input_shape=(input_dim,)))
model_clf.add(Activation('relu'))
model_clf.add(Dropout(0.3))
model_clf.add(Dense(64))
model_clf.add(Activation('relu'))
model_clf.add(Dropout(0.3))
model_clf.add(Dense(32))
model_clf.add(Activation('relu'))
model_clf.add(Dropout(0.2))
model_clf.add(Dense(1))
model_clf.add(Activation('sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model_clf.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [17]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True
)

In [18]:
model_clf.fit(
    X_train_processed, y_train,
    epochs=500,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.6483 - loss: 0.6112 - val_accuracy: 0.7306 - val_loss: 0.4918
Epoch 2/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7353 - loss: 0.4897 - val_accuracy: 0.7494 - val_loss: 0.4503
Epoch 3/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7545 - loss: 0.4576 - val_accuracy: 0.7544 - val_loss: 0.4476
Epoch 4/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7655 - loss: 0.4457 - val_accuracy: 0.7550 - val_loss: 0.4363
Epoch 5/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7655 - loss: 0.4361 - val_accuracy: 0.7500 - val_loss: 0.4427
Epoch 6/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7589 - loss: 0.4349 - val_accuracy: 0.7563 - val_loss: 0.4340
Epoch 7/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7686 - loss: 0.4269 - val_accuracy: 0.7669 - val_loss: 0.4268
Epoch 8/500
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7622 - loss: 0.4255 - val_accu

In [19]:
y_pred_nn_prob = model_clf.predict(X_test_processed).flatten()
y_pred_nn_clf = (y_pred_nn_prob >= 0.5).astype(int)

print('Neural Network Classification:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_nn_clf):.3f}')
print(f'F1 Score: {f1_score(y_test, y_pred_nn_clf):.3f}')
print(f'AUC:      {roc_auc_score(y_test, y_pred_nn_prob):.3f}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Neural Network Classification:
Accuracy: 0.779
F1 Score: 0.770
AUC:      0.887


Classification model results and comparison

In [20]:
results = {
    'Logistic Regression': {
        'acc': accuracy_score(y_test, y_pred_lr),
    },
    'Random Forest': {
        'acc': accuracy_score(y_test, y_pred_rf),
    },
    'Gradient Boosting': {
        'acc': accuracy_score(y_test, y_pred_gb),
    },
    'Neural Network': {
        'acc': accuracy_score(y_test, y_pred_nn_clf)
    }
}

for name, r in results.items():
    print(f'{name:<25} {r["acc"]:>10.3f}')

best_name = max(results, key=lambda k: results[k]['acc'])
print(f'\nBest model: {best_name} (accuracy: {results[best_name]["acc"]:.3f})')

Logistic Regression            0.769
Random Forest                  0.772
Gradient Boosting              0.786
Neural Network                 0.779

Best model: Gradient Boosting (accuracy: 0.786)
